In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [52]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

In [38]:
loader = PyPDFLoader("C:/Users/rpaul/Downloads/12+Rules+to+Learn+to+Code+[2nd+Edition]+2022.pdf")

In [39]:
pdf = loader.load()
len(pdf)

51

In [42]:
splitters = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 200 )

In [43]:
splits = splitters.split_documents(pdf)
len(splits)

70

load->split->embed->add in vector db

In [44]:
embedding = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001")

In [45]:
vector_store = Chroma.from_documents(
    documents = splits,
    embedding = embedding,
    persist_directory="./vector_db"
)

In [47]:
query = "development"

In [48]:
data = vector_store.similarity_search(query=query)
data

[Document(id='a311ee62-8d56-4d1e-ba5a-6c42c5b9c071', metadata={'creator': 'Google', 'page': 6, 'total_pages': 51, 'source': 'C:/Users/rpaul/Downloads/12+Rules+to+Learn+to+Code+[2nd+Edition]+2022.pdf', 'title': '12 Rules to Learn to Code New Template 2021', 'creationdate': '', 'producer': 'PyPDF', 'page_label': '7'}, page_content='RULE TWO\nCode for a Purpose\n2'),
 Document(id='31d05438-d860-4ae9-a023-2c81aca1c353', metadata={'creator': 'Google', 'page_label': '45', 'title': '12 Rules to Learn to Code New Template 2021', 'creationdate': '', 'total_pages': 51, 'page': 44, 'source': 'C:/Users/rpaul/Downloads/12+Rules+to+Learn+to+Code+[2nd+Edition]+2022.pdf', 'producer': 'PyPDF'}, page_content='Learning to code is a bit like \ngoing to the gym.\nwhat is butter, we’re not making Skynet here, so let’s just \nstick to the practical things. There are three things we need \nthe robot to do:\nPick up and arrange the piece of toast in the ideal buttering \nposition.\nPick up a serving of butter.

In [50]:
context = ""
for doc in data:
    context+= doc.page_content + "\n"
print(context)

RULE TWO
Code for a Purpose
2
Learning to code is a bit like 
going to the gym.
what is butter, we’re not making Skynet here, so let’s just 
stick to the practical things. There are three things we need 
the robot to do:
Pick up and arrange the piece of toast in the ideal buttering 
position.
Pick up a serving of butter.
Place butter on toast with decent coverage (this is the part I 
find most difficult).
Next, you break each module down even further. In the 
process, you can think about alternate ways of solving the 
problem. For example, does the robot need to “spread” the 
butter? Or can it just melt the butter onto the toast? Does it 
need to learn to pick up a knife? Or can it have some sort of 
inbuilt knife-arm, like some sort of prison shiv pirate?
The more that you break down problems and define the 
issue that you’re trying to solve, the easier it is to package 
your code into bite-sized chunks. The simpler the chunk, 
the easier it is to tackle. 
So the next time that you’re

In [51]:
llm = ChatGoogleGenerativeAI(model = "gemini-3.6-flash")

chaining : context_generated | prompt | llm | strparser

In [55]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context+=doc.page_content + "\n"
    
    return {
        "context": context,
        "question": query
    }
        

In [53]:
prompt = PromptTemplate.from_template("""
                                      You are a helpful assistant and provide answers based on the conext provided 
                                      and if you dont know the answer say "I dont know darling" 
                                      Content : {context}
                                      Question : {question}
                                      """)

In [56]:
rag = get_context | prompt | llm

In [59]:
res = rag.invoke("how to be consistent in coding daily")

In [60]:
res.text

"Based on the context provided, here is how you can be consistent in coding daily:\n\n1. **Develop a Habit using the Calendar/Streak Trick:** Commit to coding daily for a month. Use a monthly calendar and draw a line through each day you practice coding, extending the line each consecutive day. The motivation not to break that continuous line will help keep you on track.\n2. **Use the 20-Minute Trick:** The moment you get home and enter a new environment, tell yourself you are only going to do **20 minutes** of coding practice. Your brain won't perceive 20 minutes as a huge effort, making it much easier to overcome the difficulty of task-switching.\n3. **Take Advantage of Inertia:** Once you start those 20 minutes, human inertia kicks in. You will likely get absorbed in the project and naturally end up coding for an hour or more."